# UAV Training - YOLO Detection (Auto-Tuned)

Tek hucre calistirin, her sey otomatik.

In [ ]:
VERSION        = '0.6.0'
REPO_URL       = 'https://github.com/CELEBI-AIA/AIA-training.git'
REPO_BRANCH    = 'main'
DRIVE_DATASET  = '/content/drive/MyDrive/AIA/datasets.tar.gz'
LOCAL_CACHE    = '/content/datasets_local'
DRIVE_RUNS     = '/content/drive/MyDrive/AIA/runs'
DRIVE_UPLOAD   = '/content/drive/MyDrive/AIA'
TRAIN_SCRIPT   = 'uav_training/train.py'

import subprocess, sys, os, glob, time, gc
from datetime import datetime

os.environ['PYTHONUNBUFFERED'] = '1'

def _run(cmd, check=True, **kw):
    sys.stdout.flush(); sys.stderr.flush()
    r = subprocess.run(cmd, shell=True, check=check,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kw)
    if r.stdout:
        sys.stdout.write(r.stdout)
        sys.stdout.flush()
    return r

def _banner(msg):
    sep = '=' * 60
    print(f'\n{sep}\n  {msg}\n{sep}', flush=True)

print(f'\n🛰️  UAV Training Bootstrap v{VERSION}', flush=True)
print(f'    Repo: {REPO_URL} ({REPO_BRANCH})\n', flush=True)

# === 0. Pre-Flight Cleanup ===
_banner('0/8 -- Pre-flight cleanup')
print('  Killing stale processes...', flush=True)
_run("pkill -9 -f 'uav_training/train.py' 2>/dev/null || true", check=False)
_run("pkill -9 -f 'yolo' 2>/dev/null || true", check=False)
_run("pkill -9 -f 'build_dataset.py' 2>/dev/null || true", check=False)
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print('  GPU VRAM cache cleared', flush=True)
except Exception:
    pass
gc.collect()
REPO_DIR = '/content/repo'
if os.path.isdir(REPO_DIR):
    _run(f'find {REPO_DIR} -type d -name __pycache__ -exec rm -rf {{}} + 2>/dev/null || true', check=False)
    _run(f'find {REPO_DIR}/artifacts -name "*.cache" -delete 2>/dev/null || true', check=False)
    print('  Stale caches cleared', flush=True)
_run('rm -rf /tmp/pip-* /tmp/torch_* 2>/dev/null || true', check=False)
print('  Done\n', flush=True)

# === 1. Mount Google Drive ===
_banner('1/8 -- Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
if not os.path.isfile(DRIVE_DATASET):
    raise FileNotFoundError(f'Dataset not found: {DRIVE_DATASET}')
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(DRIVE_UPLOAD, exist_ok=True)
tar_size = os.path.getsize(DRIVE_DATASET) / (1024**3)
print(f'  Archive: {tar_size:.2f} GB', flush=True)

# === 2. Repository ===
_banner('2/8 -- Setting up repository')
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('  Pulling latest...', flush=True)
    _run(f'git -C {REPO_DIR} fetch --all')
    _run(f'git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}')
else:
    print(f'  Cloning {REPO_URL}...', flush=True)
    _run(f'git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
_run(f'git -C {REPO_DIR} log --oneline -1')
print('  Repo ready', flush=True)

# === 3. Dependencies ===
_banner('3/8 -- Installing dependencies')
req = os.path.join(REPO_DIR, 'requirements.txt')
if os.path.isfile(req):
    _run(f'{sys.executable} -m pip install --progress-bar on -r {req}')
_run(f'{sys.executable} -m pip install --progress-bar on ultralytics psutil', check=False)
print('  Done', flush=True)

# === 4. Dataset Extraction (MAX SPEED) ===
_banner('4/8 -- Dataset extraction (MAX SPEED)')
_run('apt-get update -qq && apt-get install -y pigz mbuffer 2>&1 | tail -3', check=False)
os.makedirs(LOCAL_CACHE, exist_ok=True)
NCPU = os.cpu_count() or 2
MARKER = os.path.join(LOCAL_CACHE, '.done')
TAR_FAST = '--no-same-owner --no-same-permissions -b 512'
HAS_MBUFFER = os.system('which mbuffer >/dev/null 2>&1') == 0
existing = sum(len(f) for _,_,f in os.walk(LOCAL_CACHE))
t0 = time.time()

if os.path.isfile(MARKER):
    print(f'  Cache complete ({existing} files) -- SKIP', flush=True)
elif existing > 5000:
    print(f'  Dataset detected ({existing} files) -- SKIP', flush=True)
    open(MARKER, 'w').write('1')
elif existing > 100:
    print(f'  Partial ({existing}) -- incremental', flush=True)
    _run(f'pigz -d -c -p {NCPU} "{DRIVE_DATASET}" '
         f'| tar -xf - -C "{LOCAL_CACHE}" {TAR_FAST} '
         f'--skip-old-files --checkpoint=10000 '
         f'--checkpoint-action=echo="  %u kontrol edildi"')
    open(MARKER, 'w').write('1')
else:
    if HAS_MBUFFER:
        print(f'  Drive -> mbuffer(64MB) -> pigz({NCPU}) -> tar', flush=True)
        _run(f'mbuffer -i "{DRIVE_DATASET}" -m 64M -q '
             f'| pigz -d -c -p {NCPU} '
             f'| tar -xf - -C "{LOCAL_CACHE}" {TAR_FAST} '
             f'--checkpoint=10000 '
             f'--checkpoint-action=echo="  %u cikarildi"')
    else:
        print(f'  pigz({NCPU}) -> tar', flush=True)
        _run(f'pigz -d -c -p {NCPU} "{DRIVE_DATASET}" '
             f'| tar -xf - -C "{LOCAL_CACHE}" {TAR_FAST} '
             f'--checkpoint=10000 '
             f'--checkpoint-action=echo="  %u cikarildi"')
    open(MARKER, 'w').write('1')

final = sum(len(f) for _,_,f in os.walk(LOCAL_CACHE))
print(f'  Done in {time.time()-t0:.1f}s -- {final} files', flush=True)
repo_ds = os.path.join(REPO_DIR, 'datasets')
if os.path.islink(repo_ds) or os.path.isdir(repo_ds):
    _run(f'rm -rf "{repo_ds}"')
os.symlink(LOCAL_CACHE, repo_ds)
_run('df -h /content | tail -1')

# === 5. Hardware Detection ===
_banner('5/8 -- Auto-detecting hardware')
os.environ['UAV_PROJECT_DIR'] = DRIVE_RUNS
os.environ['DRIVE_UPLOAD_DIR'] = DRIVE_UPLOAD
_run(f'yolo settings runs_dir="{DRIVE_RUNS}"', check=False)
_run('nvidia-smi', check=False)
try:
    import torch as _t
    if _t.cuda.is_available():
        _p = _t.cuda.get_device_properties(0)
        print(f'  GPU: {_p.name}  |  VRAM: {_p.total_memory / 1024**3:.1f} GB', flush=True)
    del _t, _p
except Exception:
    pass

# === 6. Training (with live log tee) ===
_banner('6/8 -- Starting training')

def find_ckpt(d):
    c = glob.glob(os.path.join(d, '**', 'weights', 'last.pt'), recursive=True)
    return max(c, key=os.path.getmtime) if c else None

ckpt = find_ckpt(DRIVE_RUNS)
script = os.path.join(REPO_DIR, TRAIN_SCRIPT)
if not os.path.isfile(script):
    raise FileNotFoundError(f'Not found: {script}')

log_dir = os.path.join(DRIVE_UPLOAD, 'logs')
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, datetime.now().strftime('log_%Y-%m-%d_%H-%M.txt'))

if ckpt:
    print(f'  Resuming: {ckpt}', flush=True)
    cmd = f'{sys.executable} -u "{script}" --resume'
else:
    print('  Fresh training', flush=True)
    cmd = f'{sys.executable} -u "{script}"'

print(f'  Log: {log_path}', flush=True)
os.chdir(REPO_DIR)

# Real-time tee: Popen + os.read -> screen + log file
proc = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'}
)
with open(log_path, 'wb') as lf:
    fd = proc.stdout.fileno()
    while True:
        data = os.read(fd, 8192)
        if not data:
            break
        sys.stdout.write(data.decode('utf-8', errors='replace'))
        sys.stdout.flush()
        lf.write(data)
        lf.flush()
ec = proc.wait()
print(f'\n  Exit code: {ec}  |  Log: {log_path}', flush=True)

# === 7. Summary ===
_banner('7/8 -- Summary')
models_dir = os.path.join(DRIVE_UPLOAD, 'models')
if os.path.isdir(models_dir):
    folders = sorted([d for d in os.listdir(models_dir) if os.path.isdir(os.path.join(models_dir, d))])
    for mf in folders[-5:]:
        n = len(os.listdir(os.path.join(models_dir, mf)))
        print(f'  {mf} ({n} files)')
for bf in sorted(glob.glob(os.path.join(DRIVE_UPLOAD, 'best_*.pt'))):
    sz = os.path.getsize(bf) / (1024 * 1024)
    print(f'  -> {os.path.basename(bf)} ({sz:.1f} MB)')
for lf in sorted(glob.glob(os.path.join(log_dir, 'log_*.txt')))[-3:]:
    sz = os.path.getsize(lf) / 1024
    print(f'  Log: {os.path.basename(lf)} ({sz:.0f} KB)')
_banner('Done -- outputs on Google Drive')